In [1]:
import numpy as np


In [2]:
data = np.load("dataset_mc_ml_v1.npz")

X = data["X"]
y = data["y"]

feature_names = data["feature_names"]
target_names = data["target_names"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("Targets:", target_names)


X shape: (200, 12)
y shape: (200, 2)
Features: ['poa_mean' 'poa_std' 'poa_amp' 'M_bar_mean' 'M_bar_std' 'M_bar_amp'
 'n_mean' 'n_std' 'n_amp' 'churn_mean' 'churn_std' 'churn_amp']
Targets: ['theta_cost' 'theta_cong']


In [3]:
def train_val_test_split(X, y, train_frac=0.7, val_frac=0.15, seed=42):
    rng = np.random.default_rng(seed)

    N = X.shape[0]
    idx = np.arange(N)
    rng.shuffle(idx)

    train_end = int(train_frac * N)
    val_end = int((train_frac + val_frac) * N)

    train_idx = idx[:train_end]
    val_idx = idx[train_end:val_end]
    test_idx = idx[val_end:]

    return (
        X[train_idx], y[train_idx],
        X[val_idx], y[val_idx],
        X[test_idx], y[test_idx]
    )


In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X, y)

print(X_train.shape, X_val.shape, X_test.shape)


(140, 12) (30, 12) (30, 12)


In [5]:
# media e std SOLO sul training set
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)

# per sicurezza
X_std[X_std == 0] = 1.0


In [6]:
def standardize(X, mean, std):
    return (X - mean) / std


In [7]:
X_train_std = standardize(X_train, X_mean, X_std)
X_val_std   = standardize(X_val,   X_mean, X_std)
X_test_std  = standardize(X_test,  X_mean, X_std)


In [8]:
print("Train mean (≈0):", X_train_std.mean(axis=0))
print("Train std  (≈1):", X_train_std.std(axis=0))


Train mean (≈0): [ 4.81261856e-15 -2.69328211e-16  3.13055633e-16 -1.43020516e-15
  5.53128971e-15  2.94526308e-15  2.86596144e-15 -1.55431223e-16
 -2.48372751e-15 -1.47818266e-15 -3.48927236e-17 -1.61775355e-15]
Train std  (≈1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [9]:
np.savez(
    "dataset_mc_ml_v1_prepared.npz",
    X_train=X_train_std,
    y_train=y_train,
    X_val=X_val_std,
    y_val=y_val,
    X_test=X_test_std,
    y_test=y_test,
    X_mean=X_mean,
    X_std=X_std,
    feature_names=feature_names,
    target_names=target_names
)
